In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

## Step 2 — Set Project Paths

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
REPORT_PATH = PROJECT_ROOT / "reports"
SQL_PATH = PROJECT_ROOT / "sql"
DB_PATH = PROJECT_ROOT / "bluestock_mf.db"

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH.mkdir(parents=True, exist_ok=True)
SQL_PATH.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("Raw:", RAW_PATH.exists())

Project Root: d:\Mutual_Fund_Analytics_IPYNB_Day1
Raw: True


## Step 3 — Clean NAV History

In [3]:
nav = pd.read_csv(RAW_PATH / "02_nav_history.csv")

nav["date"] = pd.to_datetime(nav["date"], errors="coerce")
nav["nav"] = pd.to_numeric(nav["nav"], errors="coerce")

nav = nav.drop_duplicates()
nav = nav.sort_values(["amfi_code", "date"])

invalid_nav_count = (nav["nav"] <= 0).sum()
nav = nav[nav["nav"] > 0]

nav["nav"] = nav.groupby("amfi_code")["nav"].ffill()
nav["daily_return_pct"] = nav.groupby("amfi_code")["nav"].pct_change() * 100

nav.to_csv(PROCESSED_PATH / "clean_nav.csv", index=False)

print("clean_nav.csv created:", nav.shape)
display(nav.head())

clean_nav.csv created: (46000, 4)


,amfi_code,date,nav,daily_return_pct
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-1.030568
5752,100016,2022-01-05,521.7239,1.286515
5753,100016,2022-01-06,515.7880,-1.137747
5754,100016,2022-01-07,515.1639,-0.120999


## Step 4 — Clean Investor Transactions

In [4]:
txn = pd.read_csv(RAW_PATH / "08_investor_transactions.csv")

txn["transaction_date"] = pd.to_datetime(txn["transaction_date"], errors="coerce")
txn["amount_inr"] = pd.to_numeric(txn["amount_inr"], errors="coerce")

txn["transaction_type"] = (
    txn["transaction_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "sip": "SIP",
        "lumpsum": "Lumpsum",
        "lump sum": "Lumpsum",
        "redemption": "Redemption",
        "redeem": "Redemption"
    })
)

if "kyc_status" in txn.columns:
    txn["kyc_status"] = txn["kyc_status"].astype(str).str.strip().str.title()

invalid_amount_count = (txn["amount_inr"] <= 0).sum()
txn = txn[txn["amount_inr"] > 0]
txn = txn.drop_duplicates()

txn.to_csv(PROCESSED_PATH / "clean_transactions.csv", index=False)

print("clean_transactions.csv created:", txn.shape)
display(txn.head())

clean_transactions.csv created: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


## Step 5 — Clean Scheme Performance

In [5]:
perf = pd.read_csv(RAW_PATH / "07_scheme_performance.csv")

num_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct",
    "alpha",
    "beta",
    "sharpe_ratio",
    "sortino_ratio",
    "std_dev_ann_pct",
    "max_drawdown_pct",
    "aum_crore",
    "expense_ratio_pct",
    "morningstar_rating"
]

for col in num_cols:
    if col in perf.columns:
        perf[col] = pd.to_numeric(perf[col], errors="coerce")

perf["negative_sharpe_flag"] = perf["sharpe_ratio"] < 0
perf["expense_ratio_anomaly_flag"] = ~perf["expense_ratio_pct"].between(0.1, 2.5)

perf = perf.drop_duplicates()

perf.to_csv(PROCESSED_PATH / "clean_performance.csv", index=False)

print("clean_performance.csv created:", perf.shape)
display(perf.head())

clean_performance.csv created: (40, 21)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,...,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,negative_sharpe_flag,expense_ratio_anomaly_flag
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,...,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,False,False
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,...,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,False,False
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,...,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,False,False
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,...,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,False,False
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,...,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,False,False


## Step 6 — Save Processed Files

In [6]:
copy_files = {
    "01_fund_master.csv": "dim_fund.csv",
    "03_aum_by_fund_house.csv": "fact_aum.csv",
    "04_monthly_sip_inflows.csv": "fact_sip_industry.csv",
    "05_category_inflows.csv": "fact_category_inflows.csv",
    "06_industry_folio_count.csv": "fact_folio_count.csv",
    "09_portfolio_holdings.csv": "fact_portfolio.csv",
    "10_benchmark_indices.csv": "fact_benchmark_indices.csv"
}

for raw_file, clean_file in copy_files.items():
    df = pd.read_csv(RAW_PATH / raw_file).drop_duplicates()
    df.to_csv(PROCESSED_PATH / clean_file, index=False)
    print(clean_file, df.shape)

dim_fund.csv (40, 15)
fact_aum.csv (90, 5)
fact_sip_industry.csv (48, 6)
fact_category_inflows.csv (144, 3)
fact_folio_count.csv (21, 6)
fact_portfolio.csv (322, 8)
fact_benchmark_indices.csv (8050, 3)


## Step 7 — Create SQLite Database

In [7]:
dim_date = nav[["date"]].drop_duplicates().copy()

dim_date["date"] = pd.to_datetime(dim_date["date"], errors="coerce")
dim_date = dim_date.dropna().sort_values("date")

dim_date["date_id"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["is_weekday"] = dim_date["date"].dt.dayofweek < 5

dim_date = dim_date[["date_id", "date", "year", "month", "quarter", "is_weekday"]]
dim_date.to_csv(PROCESSED_PATH / "dim_date.csv", index=False)

print("dim_date.csv created:", dim_date.shape)
display(dim_date.head())

dim_date.csv created: (1150, 6)


,date_id,date,year,month,quarter,is_weekday
5750,20220103,2022-01-03,2022,1,1,True
5751,20220104,2022-01-04,2022,1,1,True
5752,20220105,2022-01-05,2022,1,1,True
5753,20220106,2022-01-06,2022,1,1,True
5754,20220107,2022-01-07,2022,1,1,True


## Step 8-- Cleaning summary

In [10]:
summary_lines = [
    "DAY 2 DATA CLEANING SUMMARY",
    "=" * 70,
    f"clean_nav.csv: {nav.shape}",
    f"clean_transactions.csv: {txn.shape}",
    f"clean_performance.csv: {perf.shape}",
    f"dim_date.csv: {dim_date.shape}",
    f"Invalid NAV rows removed: {invalid_nav_count}",
    f"Invalid transaction amount rows removed: {invalid_amount_count}",
    f"Negative Sharpe rows: {int(perf['negative_sharpe_flag'].sum())}",
    f"Expense ratio anomaly rows: {int(perf['expense_ratio_anomaly_flag'].sum())}"
]

summary_file = REPORT_PATH / "day2_cleaning_summary.txt"
summary_file.write_text("\n".join(summary_lines), encoding="utf-8")

print("\n".join(summary_lines))

DAY 2 DATA CLEANING SUMMARY
clean_nav.csv: (46000, 4)
clean_transactions.csv: (32778, 13)
clean_performance.csv: (40, 21)
dim_date.csv: (1150, 6)
Invalid NAV rows removed: 0
Invalid transaction amount rows removed: 0
Negative Sharpe rows: 0
Expense ratio anomaly rows: 0


## Step 9 — Create schema.sql

In [11]:
schema_sql = """
DROP TABLE IF EXISTS dim_fund;
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT,
    scheme_name TEXT,
    category TEXT,
    sub_category TEXT,
    plan TEXT,
    launch_date TEXT,
    benchmark TEXT,
    expense_ratio_pct REAL,
    exit_load_pct REAL,
    min_sip_amount REAL,
    min_lumpsum_amount REAL,
    fund_manager TEXT,
    risk_category TEXT,
    sebi_category_code TEXT
);

DROP TABLE IF EXISTS dim_date;
CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY,
    date TEXT,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    is_weekday INTEGER
);

DROP TABLE IF EXISTS fact_nav;
CREATE TABLE fact_nav (
    amfi_code INTEGER,
    date TEXT,
    nav REAL,
    daily_return_pct REAL
);

DROP TABLE IF EXISTS fact_transactions;
CREATE TABLE fact_transactions (
    investor_id TEXT,
    transaction_date TEXT,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    city_tier TEXT,
    age_group TEXT,
    gender TEXT,
    annual_income_lakh REAL,
    payment_mode TEXT,
    kyc_status TEXT
);

DROP TABLE IF EXISTS fact_performance;
CREATE TABLE fact_performance (
    amfi_code INTEGER,
    scheme_name TEXT,
    fund_house TEXT,
    category TEXT,
    plan TEXT,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    benchmark_3yr_pct REAL,
    alpha REAL,
    beta REAL,
    sharpe_ratio REAL,
    sortino_ratio REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore REAL,
    expense_ratio_pct REAL,
    morningstar_rating REAL,
    risk_grade TEXT,
    negative_sharpe_flag INTEGER,
    expense_ratio_anomaly_flag INTEGER
);

DROP TABLE IF EXISTS fact_aum;
CREATE TABLE fact_aum (
    date TEXT,
    fund_house TEXT,
    aum_lakh_crore REAL,
    aum_crore REAL,
    num_schemes INTEGER
);
"""

(SQL_PATH / "schema.sql").write_text(schema_sql, encoding="utf-8")

print("schema.sql created")

schema.sql created


## Step 10 — Load into SQLite


In [12]:
conn = sqlite3.connect(DB_PATH)

table_files = {
    "dim_fund": "dim_fund.csv",
    "dim_date": "dim_date.csv",
    "fact_nav": "clean_nav.csv",
    "fact_transactions": "clean_transactions.csv",
    "fact_performance": "clean_performance.csv",
    "fact_aum": "fact_aum.csv",
    "fact_sip_industry": "fact_sip_industry.csv",
    "fact_category_inflows": "fact_category_inflows.csv",
    "fact_folio_count": "fact_folio_count.csv",
    "fact_portfolio": "fact_portfolio.csv",
    "fact_benchmark_indices": "fact_benchmark_indices.csv"
}

row_report = []

for table, file_name in table_files.items():
    df = pd.read_csv(PROCESSED_PATH / file_name)
    df.to_sql(table, conn, if_exists="replace", index=False)
    
    db_count = pd.read_sql_query(f"SELECT COUNT(*) AS total FROM {table}", conn)["total"].iloc[0]
    row_report.append(f"{table}: source={len(df)}, db={db_count}")
    print(row_report[-1])

(REPORT_PATH / "day2_database_load_summary.txt").write_text("\n".join(row_report), encoding="utf-8")

conn.close()

print("SQLite database created:", DB_PATH)

dim_fund: source=40, db=40
dim_date: source=1150, db=1150
fact_nav: source=46000, db=46000
fact_transactions: source=32778, db=32778
fact_performance: source=40, db=40
fact_aum: source=90, db=90
fact_sip_industry: source=48, db=48
fact_category_inflows: source=144, db=144
fact_folio_count: source=21, db=21
fact_portfolio: source=322, db=322
fact_benchmark_indices: source=8050, db=8050
SQLite database created: d:\Mutual_Fund_Analytics_IPYNB_Day1\bluestock_mf.db


## Step 11 — Create Analytical SQL Queries

In [13]:
queries_sql = """
-- 1. Top 5 funds by AUM
SELECT fund_house, SUM(aum_crore) AS total_aum
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;

-- 2. Average NAV by AMFI code
SELECT amfi_code, ROUND(AVG(nav), 2) AS avg_nav
FROM fact_nav
GROUP BY amfi_code
ORDER BY avg_nav DESC;

-- 3. Monthly average NAV trend
SELECT strftime('%Y-%m', date) AS month, ROUND(AVG(nav), 2) AS avg_nav
FROM fact_nav
GROUP BY month
ORDER BY month;

-- 4. Daily return summary by fund
SELECT amfi_code,
       ROUND(MIN(daily_return_pct), 2) AS min_return,
       ROUND(MAX(daily_return_pct), 2) AS max_return,
       ROUND(AVG(daily_return_pct), 2) AS avg_return
FROM fact_nav
GROUP BY amfi_code;

-- 5. Total transaction amount by transaction type
SELECT transaction_type, ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY transaction_type
ORDER BY total_amount DESC;

-- 6. Transactions count by KYC status
SELECT kyc_status, COUNT(*) AS total_transactions
FROM fact_transactions
GROUP BY kyc_status;

-- 7. Funds with expense ratio less than 1%
SELECT amfi_code, scheme_name, expense_ratio
FROM fact_performance
WHERE expense_ratio < 1
ORDER BY expense_ratio;

-- 8. Top 5 performing schemes by return percentage
SELECT amfi_code, scheme_name, return_1yr
FROM fact_performance
ORDER BY return_1yr DESC
LIMIT 5;

-- 9. Category-wise SIP inflow summary
SELECT category, ROUND(SUM(inflow_amount), 2) AS total_inflow
FROM fact_category_inflows
GROUP BY category
ORDER BY total_inflow DESC;

-- 10. Benchmark index summary
SELECT benchmark_name, ROUND(AVG(index_value), 2) AS avg_index_value
FROM fact_benchmark_indices
GROUP BY benchmark_name
ORDER BY avg_index_value DESC;
"""

(SQL_PATH / "queries.sql").write_text(queries_sql, encoding="utf-8")
print("queries.sql created successfully")

queries.sql created successfully


## Step 12 — Run Test SQL Query

In [14]:
conn = sqlite3.connect(DB_PATH)

test_query = """
SELECT fund_house, SUM(aum_crore) AS total_aum
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;
"""

test_result = pd.read_sql_query(test_query, conn)
conn.close()

print("Test query executed successfully")
display(test_result)

Test query executed successfully


,fund_house,total_aum
0,SBI Mutual Fund,8491000
1,ICICI Prudential MF,6293000
2,HDFC Mutual Fund,5732000
3,Nippon India MF,3909000
4,Kotak Mahindra MF,3502000


## Step 13 — Create Data Dictionary

In [15]:
data_dictionary = """
# Data Dictionary — Mutual Fund Analytics

## 1. dim_fund.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| amfi_code | Integer | Unique AMFI code for the mutual fund scheme | 01_fund_master.csv |
| fund_house | Text | Name of the mutual fund house | 01_fund_master.csv |
| scheme_name | Text | Name of the scheme | 01_fund_master.csv |
| category | Text | Scheme category | 01_fund_master.csv |
| sub_category | Text | Scheme sub-category | 01_fund_master.csv |

## 2. dim_date.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| date | Date | Calendar date | Derived from nav / transactions |
| year | Integer | Year extracted from date | Derived |
| month | Integer | Month extracted from date | Derived |
| day | Integer | Day extracted from date | Derived |
| month_name | Text | Month name | Derived |

## 3. clean_nav.csv / fact_nav
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| amfi_code | Integer | Fund AMFI code | 02_nav_history.csv |
| date | Date | NAV date | 02_nav_history.csv |
| nav | Float | Net Asset Value | 02_nav_history.csv |
| daily_return_pct | Float | Daily % return calculated from NAV | Derived |

## 4. clean_transactions.csv / fact_transactions
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| transaction_date | Date | Transaction date | 08_investor_transactions.csv |
| transaction_type | Text | SIP / Lumpsum / Redemption | 08_investor_transactions.csv |
| amount_inr | Float | Transaction amount in INR | 08_investor_transactions.csv |
| kyc_status | Text | KYC verification status | 08_investor_transactions.csv |

## 5. clean_performance.csv / fact_performance
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| amfi_code | Integer | Fund AMFI code | 07_scheme_performance.csv |
| scheme_name | Text | Name of scheme | 07_scheme_performance.csv |
| expense_ratio | Float | Expense ratio of the scheme | 07_scheme_performance.csv |
| return_1yr | Float | 1-year return percentage | 07_scheme_performance.csv |

## 6. fact_aum.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| fund_house | Text | Mutual fund house name | 03_aum_by_fund_house.csv |
| aum_crore | Float | Assets Under Management in crore | 03_aum_by_fund_house.csv |

## 7. fact_sip_industry.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| month | Date/Text | Month of SIP inflow | 04_monthly_sip_inflows.csv |
| sip_inflow | Float | SIP inflow amount | 04_monthly_sip_inflows.csv |

## 8. fact_category_inflows.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| category | Text | Mutual fund category | 05_category_inflows.csv |
| inflow_amount | Float | Inflow amount | 05_category_inflows.csv |

## 9. fact_folio_count.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| industry_segment | Text | Industry or fund segment | 06_industry_folio_count.csv |
| folio_count | Integer | Number of folios | 06_industry_folio_count.csv |

## 10. fact_portfolio.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| scheme_name | Text | Scheme name | 09_portfolio_holdings.csv |
| holding_name | Text | Security / holding name | 09_portfolio_holdings.csv |
| holding_weight | Float | Weight of holding in portfolio | 09_portfolio_holdings.csv |

## 11. fact_benchmark_indices.csv
| Column Name | Data Type | Description | Source |
|------------|-----------|-------------|--------|
| benchmark_name | Text | Benchmark index name | 10_benchmark_indices.csv |
| index_value | Float | Benchmark index value | 10_benchmark_indices.csv |
"""

(PROJECT_ROOT / "data_dictionary.md").write_text(data_dictionary, encoding="utf-8")
print("data_dictionary.md created successfully")

data_dictionary.md created successfully


## Step 14 — Final Day 2 Checklist

In [17]:
print("DAY 2 FINAL VALIDATION REPORT")
print("-" * 50)

files_to_check = {
    "Clean NAV data": PROCESSED_PATH / "clean_nav.csv",
    "Clean investor transactions": PROCESSED_PATH / "clean_transactions.csv",
    "Clean scheme performance": PROCESSED_PATH / "clean_performance.csv",
    "Fund dimension table": PROCESSED_PATH / "dim_fund.csv",
    "Date dimension table": PROCESSED_PATH / "dim_date.csv",
    "AUM fact table": PROCESSED_PATH / "fact_aum.csv",
    "SIP industry fact table": PROCESSED_PATH / "fact_sip_industry.csv",
    "Category inflow fact table": PROCESSED_PATH / "fact_category_inflows.csv",
    "Folio count fact table": PROCESSED_PATH / "fact_folio_count.csv",
    "Portfolio holdings fact table": PROCESSED_PATH / "fact_portfolio.csv",
    "Benchmark indices fact table": PROCESSED_PATH / "fact_benchmark_indices.csv",
    "SQLite database": DB_PATH,
    "Database schema file": SQL_PATH / "schema.sql",
    "Analytical SQL queries file": SQL_PATH / "queries.sql",
    "Data dictionary": PROJECT_ROOT / "data_dictionary.md",
    "Cleaning summary report": REPORT_PATH / "day2_cleaning_summary.txt",
    "Database load summary report": REPORT_PATH / "day2_database_load_summary.txt"
}

for description, path in files_to_check.items():
    status = "Created" if path.exists() else "Missing"
    print(f"{description}: {status}")

print("\nDay 2 work completed.")
print("Cleaned datasets, SQLite database, schema file, SQL queries, data dictionary, and summary reports are available in the project folder.")

DAY 2 FINAL VALIDATION REPORT
--------------------------------------------------
Clean NAV data: Created
Clean investor transactions: Created
Clean scheme performance: Created
Fund dimension table: Created
Date dimension table: Created
AUM fact table: Created
SIP industry fact table: Created
Category inflow fact table: Created
Folio count fact table: Created
Portfolio holdings fact table: Created
Benchmark indices fact table: Created
SQLite database: Created
Database schema file: Created
Analytical SQL queries file: Created
Data dictionary: Created
Cleaning summary report: Created
Database load summary report: Created

Day 2 work completed.
Cleaned datasets, SQLite database, schema file, SQL queries, data dictionary, and summary reports are available in the project folder.
